## **CREATE DATAHUB DOMAIN**

In [1]:
#!/usr/bin/env python3
import os
import requests
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(), override=False)

GMS = os.getenv("DATAHUB_GMS_URL", "http://datahub-gms:8080")
TOKEN = os.getenv("DATAHUB_TOKEN", "")

HEAD = {"Content-Type": "application/json"}
if TOKEN:
    HEAD["Authorization"] = f"Bearer {TOKEN}"

# list domain yang mau kamu punya di DataHub
DOMAINS = [
    {
        "id": "sales",
        "name": "Sales",
        "description": "Sales-related datasets",
    },
    {
        "id": "crm",
        "name": "CRM",
        "description": "Customer / CRM datasets",
    },
    {
        "id": "hr",
        "name": "HR",
        "description": "Human Resources datasets",
    },
    {
        "id": "catalog",
        "name": "Catalog",
        "description": "Product / catalog datasets",
    },
    {
        "id": "finance",
        "name": "Finance",
        "description": "Finance-related datasets",
    },
    {
        "id": "geo",
        "name": "Geo",
        "description": "Geographical / office datasets",
    },
]

MUTATION = """
mutation createDomain($input: CreateDomainInput!) {
  createDomain(input: $input)
}
"""

def create_domain(dom):
    payload = {
        "query": MUTATION,
        "variables": {
            "input": {
                "id": dom["id"],
                "name": dom["name"],
                "description": dom.get("description", ""),
            }
        },
    }
    r = requests.post(f"{GMS}/api/graphql", headers=HEAD, json=payload, timeout=30)
    r.raise_for_status()
    js = r.json()
    # kalau sudah ada, DataHub biasanya balikin error "DOMAIN_ALREADY_EXISTS"
    if "errors" in js:
        msg = js["errors"][0].get("message", "")
        print(f"[SKIP] {dom['id']} -> {msg}")
    else:
        print(f"[OK]   {dom['id']} created")

def main():
    print(f"Creating {len(DOMAINS)} domains to {GMS} ...")
    for d in DOMAINS:
        try:
            create_domain(d)
        except Exception as e:
            print(f"[ERR]  {d['id']} -> {e}")

if __name__ == "__main__":
    main()

Creating 6 domains to http://datahub-gms:8080 ...
[OK]   sales created
[OK]   crm created
[OK]   hr created
[OK]   catalog created
[OK]   finance created
[OK]   geo created


Setelah buat domain, baru jalankan ingestion pipeline agar bisa add datasets dengan domain yg telah dibuat.

## **CREATE DATAHUB TAGS FOR DATASET**

In [3]:
#!/usr/bin/env python3
import os
import re
import json
import requests
from typing import Dict, List, Optional, Tuple
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(), override=False)

# =========================
# CONFIG
# =========================
DATAHUB_GRAPHQL = os.getenv(
    "DATAHUB_GRAPHQL_URL", "http://datahub-gms:8080/api/graphql"
)
DATAHUB_TOKEN = os.getenv("DATAHUB_TOKEN", "")

# Kalau kamu mau paksa "jangan pernah create tag baru", set env ATTACH_ONLY=true
ATTACH_ONLY = os.getenv("ATTACH_ONLY", "false").lower() == "true"

# Enum subResourceType yang kita coba untuk kolom
SUBRESOURCE_FIELD_ENUMS = ["DATASET_FIELD", "SCHEMA_FIELD", "FIELD"]

# =========================
# DATASETS
# =========================
DATASET_URNS: Dict[str, str] = {
    "customers": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.customers,PROD)",
    "employees": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.employees,PROD)",
    "products": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.products,PROD)",
    "productlines": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.productlines,PROD)",
    "orders": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.orders,PROD)",
    "orderdetails": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.orderdetails,PROD)",
    "payments": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.payments,PROD)",
    "offices": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.offices,PROD)",
}

ALIAS_TAG_PREFIX = "alias:"
ROLE_TAG_PREFIX = "role:"

# =========================
# COLUMN ALIAS / ROLE MAPPING
# (ini sama persis kaya punyamu di atas)
# =========================
ALIAS_MAPPING: Dict[str, Dict[str, Dict[str, List[str]]]] = {
    "customers": {
        "customerNumber": {
            "aliases": [
                "customer id",
                "customer number",
                "id customer",
                "kode customer",
                "kode pelanggan",
            ],
            "role": "id",
        },
        "customerName": {
            "aliases": [
                "customer name",
                "nama customer",
                "nama pelanggan",
                "company name",
                "nama perusahaan",
                "account name",
            ],
            "role": "name",
        },
        "contactLastName": {
            "aliases": [
                "last name",
                "surname",
                "contact last name",
                "nama belakang",
                "cp last name",
                "pic last name",
            ],
            "role": "name",
        },
        "contactFirstName": {
            "aliases": [
                "first name",
                "contact first name",
                "nama depan",
                "cp first name",
                "pic first name",
            ],
            "role": "name",
        },
        "phone": {
            "aliases": [
                "telephone",
                "telepon",
                "primary phone",
                "no telp",
                "no hp",
                "phone number",
                "contact phone",
            ],
            "role": "contact",
        },
        "addressLine1": {
            "aliases": [
                "address 1",
                "alamat 1",
                "street address",
                "street address 1",
                "alamat jalan",
            ],
            "role": "address",
        },
        "addressLine2": {
            "aliases": [
                "address 2",
                "alamat 2",
                "unit / floor",
                "address line 2",
                "alamat tambahan",
            ],
            "role": "address",
        },
        "city": {
            "aliases": ["kota", "city name", "customer city"],
            "role": "geo",
        },
        "state": {
            "aliases": ["state", "provinsi", "region", "wilayah"],
            "role": "geo",
        },
        "postalCode": {
            "aliases": ["postal code", "zip code", "kode pos"],
            "role": "geo",
        },
        "country": {
            "aliases": ["country", "negara", "country name"],
            "role": "geo",
        },
        "salesRepEmployeeNumber": {
            "aliases": [
                "sales rep",
                "sales representative",
                "account manager",
                "employeeNumber (sales)",
                "assigned sales",
            ],
            "role": "fk:employees",
        },
        "creditLimit": {
            "aliases": [
                "credit limit",
                "batas kredit",
                "plafon kredit",
                "customer credit limit",
            ],
            "role": "measure",
        },
    },
    "employees": {
        "employeeNumber": {
            "aliases": [
                "employee id",
                "employee number",
                "id karyawan",
                "kode karyawan",
            ],
            "role": "id",
        },
        "lastName": {
            "aliases": ["last name", "surname", "nama belakang karyawan"],
            "role": "name",
        },
        "firstName": {
            "aliases": ["first name", "nama depan karyawan"],
            "role": "name",
        },
        "extension": {
            "aliases": ["extension", "ext", "internal phone"],
            "role": "contact",
        },
        "email": {
            "aliases": ["email", "work email", "office email", "email kantor"],
            "role": "contact",
        },
        "officeCode": {
            "aliases": ["office code", "kode office", "kode kantor"],
            "role": "fk:offices",
        },
        "reportsTo": {
            "aliases": ["reports to", "manager id", "atasan langsung", "supervisor"],
            "role": "fk:employees",
        },
        "jobTitle": {
            "aliases": ["job title", "posisi", "jabatan", "title"],
            "role": "attr",
        },
    },
    "offices": {
        "officeCode": {
            "aliases": ["office code", "kode office", "kode kantor", "office id"],
            "role": "id",
        },
        "city": {"aliases": ["kota", "city", "office city"], "role": "geo"},
        "phone": {
            "aliases": ["office phone", "telepon kantor", "phone"],
            "role": "contact",
        },
        "addressLine1": {
            "aliases": ["address 1", "alamat 1", "street address"],
            "role": "address",
        },
        "addressLine2": {
            "aliases": ["address 2", "alamat 2"],
            "role": "address",
        },
        "state": {
            "aliases": ["state", "provinsi", "region"],
            "role": "geo",
        },
        "country": {
            "aliases": ["country", "negara"],
            "role": "geo",
        },
        "postalCode": {
            "aliases": ["postal code", "zip code", "kode pos"],
            "role": "geo",
        },
        "territory": {
            "aliases": [
                "territory",
                "sales territory",
                "regional code",
                "area penjualan",
            ],
            "role": "geo",
        },
    },
    "orderdetails": {
        "orderNumber": {
            "aliases": [
                "order number",
                "nomor order",
                "id order",
                "sales order id",
            ],
            "role": "fk:orders",
        },
        "productCode": {
            "aliases": ["product code", "kode produk", "kode barang", "sku"],
            "role": "fk:products",
        },
        "quantityOrdered": {
            "aliases": [
                "quantity ordered",
                "qty ordered",
                "jumlah order",
                "order qty",
            ],
            "role": "qty",
        },
        "priceEach": {
            "aliases": ["price each", "unit price", "harga satuan", "selling price"],
            "role": "price",
        },
        "orderLineNumber": {
            "aliases": ["order line number", "line number", "urutan baris", "line seq"],
            "role": "attr",
        },
    },
    "orders": {
        "orderNumber": {
            "aliases": [
                "order number",
                "nomor order",
                "id order",
                "sales order id",
            ],
            "role": "id",
        },
        "orderDate": {
            "aliases": ["order date", "tanggal order", "tanggal pesanan"],
            "role": "date",
        },
        "requiredDate": {
            "aliases": [
                "required date",
                "tanggal dibutuhkan",
                "due date",
                "target ship date",
            ],
            "role": "date",
        },
        "shippedDate": {
            "aliases": ["shipped date", "tanggal dikirim", "tanggal shipping"],
            "role": "date",
        },
        "status": {
            "aliases": [
                "status",
                "order status",
                "status pesanan",
                "status order",
            ],
            "role": "status",
        },
        "comments": {
            "aliases": ["comments", "catatan", "notes", "order notes"],
            "role": "text",
        },
        "customerNumber": {
            "aliases": [
                "customer number",
                "customer id",
                "id customer",
                "kode customer",
            ],
            "role": "fk:customers",
        },
    },
    "payments": {
        "customerNumber": {
            "aliases": ["customer number", "customer id", "id customer"],
            "role": "fk:customers",
        },
        "checkNumber": {
            "aliases": ["check number", "payment ref", "nomor cek", "no pembayaran"],
            "role": "id",
        },
        "paymentDate": {
            "aliases": ["payment date", "tanggal bayar", "tanggal pembayaran"],
            "role": "date",
        },
        "amount": {
            "aliases": ["amount", "nominal", "jumlah bayar", "payment amount"],
            "role": "measure",
        },
    },
    "productlines": {
        "productLine": {
            "aliases": [
                "product line",
                "line produk",
                "kategori produk",
                "product category",
            ],
            "role": "name",
        },
        "textDescription": {
            "aliases": ["text description", "deskripsi teks", "deskripsi produk"],
            "role": "text",
        },
        "htmlDescription": {
            "aliases": ["html description", "deskripsi html"],
            "role": "text",
        },
        "image": {
            "aliases": ["image", "gambar", "logo"],
            "role": "binary",
        },
    },
    "products": {
        "productCode": {
            "aliases": ["product code", "kode produk", "kode barang", "sku"],
            "role": "id",
        },
        "productName": {
            "aliases": ["product name", "nama produk", "nama barang", "item name"],
            "role": "name",
        },
        "productLine": {
            "aliases": ["product line", "kategori produk", "product category"],
            "role": "fk:productlines",
        },
        "productScale": {
            "aliases": ["product scale", "skala produk", "scale", "rasio model"],
            "role": "attr",
        },
        "productVendor": {
            "aliases": ["product vendor", "vendor", "supplier", "manufacturer"],
            "role": "attr",
        },
        "productDescription": {
            "aliases": ["product description", "deskripsi produk", "keterangan barang"],
            "role": "text",
        },
        "quantityInStock": {
            "aliases": [
                "qty in stock",
                "quantity in stock",
                "stok",
                "jumlah stok",
                "on hand",
            ],
            "role": "qty",
        },
        "buyPrice": {
            "aliases": ["buy price", "purchase price", "harga beli", "cost price"],
            "role": "price",
        },
        "MSRP": {
            "aliases": ["list price", "harga list", "msrp", "retail price"],
            "role": "price",
        },
    },
}

# =========================
# GRAPHQL HELPERS
# =========================
def _headers() -> Dict[str, str]:
    h = {"Content-Type": "application/json"}
    if DATAHUB_TOKEN:
        h["Authorization"] = f"Bearer {DATAHUB_TOKEN}"
    else:
        h["X-DataHub-Actor"] = "urn:li:corpuser:admin"
    return h


def gql(query: str, variables: dict) -> dict:
    resp = requests.post(
        DATAHUB_GRAPHQL,
        headers=_headers(),
        json={"query": query, "variables": variables},
        timeout=30,
    )
    if resp.status_code != 200:
        raise RuntimeError(f"GraphQL error {resp.status_code}: {resp.text}")
    data = resp.json()
    if "errors" in data and data["errors"]:
        raise RuntimeError(f"GraphQL returned errors: {data['errors']}")
    return data


# =========================
# TAG UTILITIES
# =========================
_BAD_CHARS = re.compile(r"[^A-Za-z0-9._-]")
TAG_CACHE: Dict[str, bool] = {}  # tag_id -> exists


def make_tag_id(display_name: str) -> str:
    s = display_name.replace(":", ".").replace(" ", "_")
    s = _BAD_CHARS.sub("_", s).strip("._-") or "tag"
    return s


def list_tags_by_id(tag_id: str) -> int:
    q = """
    query listTags($input: ListTagsInput!) {
      listTags(input: $input) { total }
    }
    """
    res = gql(
        q,
        {"input": {"start": 0, "count": 1, "filter": {"field": "id", "value": tag_id}}},
    )
    return int(((res.get("data") or {}).get("listTags") or {}).get("total") or 0)


def tag_exists(tag_id: str) -> bool:
    if tag_id in TAG_CACHE:
        return TAG_CACHE[tag_id]
    try:
        exists = list_tags_by_id(tag_id) > 0
    except Exception:
        exists = False
    TAG_CACHE[tag_id] = exists
    return exists


def ensure_tag(display_name: str) -> str:
    tag_id = make_tag_id(display_name)
    urn = f"urn:li:tag:{tag_id}"

    # sudah pernah dicek?
    if tag_id in TAG_CACHE and TAG_CACHE[tag_id]:
        return urn

    # cek ke server sekali
    if tag_exists(tag_id):
        return urn

    if ATTACH_ONLY:
        print(
            f"[INFO] ATTACH_ONLY on: assuming tag exists (id='{tag_id}', name='{display_name}')"
        )
        TAG_CACHE[tag_id] = True
        return urn

    # create
    mutation = """
    mutation createTag($input: CreateTagInput!) {
      createTag(input: $input)
    }
    """
    variables = {
        "input": {
            "id": tag_id,
            "name": display_name,
            "description": f"Autocreated by classicmodels (tag: {display_name})",
        }
    }
    try:
        gql(mutation, variables)
        TAG_CACHE[tag_id] = True
        print(f"[CREATE] tag {display_name} ({tag_id})")
    except RuntimeError as e:
        msg = str(e).lower()
        if "already exists" in msg:
            TAG_CACHE[tag_id] = True
            print(f"[INFO] tag {display_name} ({tag_id}) already exists")
        else:
            print(f"[WARN] cannot create tag '{display_name}' (id='{tag_id}'): {e}")
            TAG_CACHE[tag_id] = False
    return urn


def _try_addTags(
    resource_urn: str,
    tag_urns: List[str],
    sub_resource: Optional[str],
    sub_resource_type: Optional[str],
):
    mutation = """
    mutation addTags($input: AddTagsInput!) {
      addTags(input: $input)
    }
    """
    variables = {
        "input": {
            "resourceUrn": resource_urn,
            "tagUrns": tag_urns,
            "subResource": sub_resource,
            "subResourceType": sub_resource_type,
        }
    }
    return gql(mutation, variables)


def add_tags_column(
    resource_urn: str,
    tag_urns: List[str],
    sub_resource: str,
):
    """
    Khusus kolom: coba semua enum, kalau gagal fallback per-tag.
    """
    if not tag_urns:
        return

    last_err = None
    for enum_name in SUBRESOURCE_FIELD_ENUMS:
        try:
            _try_addTags(resource_urn, tag_urns, sub_resource, enum_name)
            return
        except RuntimeError as e:
            last_err = e

    # fallback per tag
    mutation_single = """
    mutation addTag($input: TagAssociationInput!) {
      addTag(input: $input)
    }
    """
    for t in tag_urns:
        attached = False
        for enum_name in SUBRESOURCE_FIELD_ENUMS:
            try:
                gql(
                    mutation_single,
                    {
                        "input": {
                            "resourceUrn": resource_urn,
                            "tagUrn": t,
                            "subResource": sub_resource,
                            "subResourceType": enum_name,
                        }
                    },
                )
                attached = True
                break
            except RuntimeError as e:
                last_err = e
        if not attached and last_err:
            raise last_err


def build_field_tag_urns(
    aliases: Optional[List[str]],
    role: Optional[str],
) -> List[str]:
    urns: List[str] = []
    for al in aliases or []:
        urns.append(ensure_tag(f"{ALIAS_TAG_PREFIX}{al}"))
    if role:
        urns.append(ensure_tag(f"{ROLE_TAG_PREFIX}{role}"))

    # de-dup
    out, seen = [], set()
    for u in urns:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out


def gather_all_display_tags_column_only() -> List[Tuple[str, str]]:
    """
    Beda dari versi awal: kita TIDAK ambil dari TABLE_BASE_TAG,
    jadi yang dibuat hanya alias:* dan role:*.
    """
    tags: List[str] = []
    for _, fields in ALIAS_MAPPING.items():
        for _, cfg in fields.items():
            for al in cfg.get("aliases", []):
                tags.append(f"{ALIAS_TAG_PREFIX}{al}")
            role = cfg.get("role")
            if role:
                tags.append(f"{ROLE_TAG_PREFIX}{role}")
    out, seen = [], set()
    for t in tags:
        tid = make_tag_id(t)
        if tid not in seen:
            seen.add(tid)
            out.append((t, tid))
    return out


def print_tag_manifest():
    print("\n[MANIFEST] Column Tags (display_name -> id / urn):")
    for display, tid in gather_all_display_tags_column_only():
        print(f"  {display:<40} -> {tid:<40}  urn:li:tag:{tid}")
    print("")


# =========================
# MAIN
# =========================
def main():
    print_tag_manifest()
    print(f"\n[INFO] ATTACH_ONLY = {ATTACH_ONLY}\n")

    # 1) Column-level tags saja
    for dataset_name, fields in ALIAS_MAPPING.items():
        dataset_urn = DATASET_URNS[dataset_name]
        for field_path, cfg in fields.items():
            aliases = cfg.get("aliases", [])
            role = cfg.get("role")
            tag_urns = build_field_tag_urns(aliases, role)

            if not tag_urns:
                continue

            try:
                add_tags_column(
                    resource_urn=dataset_urn,
                    tag_urns=tag_urns,
                    sub_resource=field_path,
                )
                print(
                    f"[OK] {dataset_name}.{field_path} -> {len(tag_urns)} tags attached"
                )
            except RuntimeError as e:
                print(
                    f"[WARN] {dataset_name}.{field_path} -> attach failed: {e}"
                )

    print("[DONE] column-only enrichment finished.")


if __name__ == "__main__":
    main()


[MANIFEST] Column Tags (display_name -> id / urn):
  alias:customer id                        -> alias.customer_id                         urn:li:tag:alias.customer_id
  alias:customer number                    -> alias.customer_number                     urn:li:tag:alias.customer_number
  alias:id customer                        -> alias.id_customer                         urn:li:tag:alias.id_customer
  alias:kode customer                      -> alias.kode_customer                       urn:li:tag:alias.kode_customer
  alias:kode pelanggan                     -> alias.kode_pelanggan                      urn:li:tag:alias.kode_pelanggan
  role:id                                  -> role.id                                   urn:li:tag:role.id
  alias:customer name                      -> alias.customer_name                       urn:li:tag:alias.customer_name
  alias:nama customer                      -> alias.nama_customer                       urn:li:tag:alias.nama_customer
  alias:n

In [ ]:
#!/usr/bin/env python3
import os
import time
import requests
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(), override=False)

GMS   = os.environ.get("DATAHUB_GMS_URL", "http://datahub-gms:8080")
TOKEN = os.environ.get("DATAHUB_TOKEN")
HEAD  = {"Content-Type": "application/json"}
if TOKEN:
    HEAD["Authorization"] = f"Bearer {TOKEN}"
else:
    HEAD["X-DataHub-Actor"] = "urn:li:corpuser:admin"

# --- di output-mu ketemu pola: alias.xxx (di urn/id) dan alias:xxx (di name)
# jadi kita bikin prefix yang "bebas pemisah"
BASE_PREFIXES = ("alias", "role", "dom")

DRY_RUN  = False
PAGE_SZ  = 200

def gql(q, v):
    r = requests.post(
        f"{GMS}/api/graphql",
        headers=HEAD,
        json={"query": q, "variables": v},
        timeout=30,
    )
    r.raise_for_status()
    js = r.json()
    if "errors" in js:
        raise RuntimeError(js["errors"])
    return js["data"]

def list_all_tags():
    """Return list of (urn, id, name) for all tags."""
    out, start = [], 0
    q = """
    query($start:Int!, $count:Int!) {
      search(input:{type:TAG, query:"", start:$start, count:$count}) {
        total
        searchResults {
          entity {
            urn
            ... on Tag {
              urn
              properties { name }
            }
          }
        }
      }
    }"""
    while True:
        d = gql(q, {"start": start, "count": PAGE_SZ})
        sr = d["search"]["searchResults"]
        if not sr:
            break
        for r in sr:
            ent = r["entity"]
            urn = ent["urn"]
            # di kasusmu: urn:li:tag:alias.address_1
            tag_id = urn.split("urn:li:tag:")[-1]
            props = ent.get("properties") or {}
            name  = props.get("name")  # contoh: "alias:address 1"
            out.append((urn, tag_id, name))
        start += PAGE_SZ
        if start >= d["search"]["total"]:
            break
    return out

def hard_delete(urn):
    r = requests.post(
        f"{GMS}/entities?action=delete",
        headers=HEAD,
        json={"urn": urn, "hardDelete": True},
        timeout=30,
    )
    r.raise_for_status()
    return r.json()

def match_prefix(tag_id: str, name: str) -> bool:
    """
    Kembalikan True kalau tag ini masuk salah satu BASE_PREFIXES.
    Kita normalize:
      - lowercase
      - ganti ":" jadi "."
    lalu cek prefix.
    """
    candidates = []

    if tag_id:
        candidates.append(tag_id.lower())
        candidates.append(tag_id.lower().replace(":", "."))
    if name:
        candidates.append(name.lower())
        candidates.append(name.lower().replace(":", "."))

    for cand in candidates:
        for base in BASE_PREFIXES:
            if cand.startswith(base + "."):
                return True
    return False

def main():
    all_tags = list_all_tags()

    print("== tags ditemukan dari GMS ==")
    for urn, tid, name in all_tags:
        print(f"- {urn}  (id={tid}, name={name})")

    # filter
    targets = []
    for urn, tid, name in all_tags:
        if match_prefix(tid, name):
            targets.append((urn, tid, name))

    if not targets:
        # kasih info prefix yang sekarang dipakai
        expanded = [f"{p}. / {p}:" for p in BASE_PREFIXES]
        print("\nNo matching tags with prefixes (normalized):", ", ".join(expanded))
        return

    print(f"\nFound {len(targets)} tag(s) to remove:")
    for urn, tid, name in targets:
        print(f"  - {urn}  (name={name}, id={tid})")

    if DRY_RUN:
        print("\n[DRY RUN] Set DRY_RUN=False to actually delete.")
        return

    print("\nDeleting...")
    ok = 0
    for urn, tid, name in targets:
        try:
            hard_delete(urn)
            ok += 1
            print(f"  OK  {urn}  ({name})")
            time.sleep(0.05)
        except Exception as e:
            print(f"  FAIL {urn} -> {e}")

    print(f"\nDone. Deleted {ok}/{len(targets)} tags.")

if __name__ == "__main__":
    main()